# Demo 04 — RAG with DSPy: Build, Measure, Optimize

The runnable notebook behind **§3 (build one)**, **§4.6 (did retrieval help?)**, and
**§8.7 (MIPROv2)** of *Chapter 04 — Inference-Time Retrieval Patterns*.

We build a question-answering system over `ragqa_arena_tech` — a corpus of community developer
Q&A answers — with [DSPy](https://dspy.ai/) and Anthropic's Claude, measure whether retrieval
actually moves answer quality, and finally optimize the pipeline's prompt automatically.

Adapted from the [DSPy RAG tutorial](https://dspy.ai/tutorials/rag/) to run on Claude.

**What you need:** an `ANTHROPIC_API_KEY` in a `.env` file (on Colab, in Colab Secrets — sidebar
key icon — instead). Retrieval runs locally on CPU (`sentence-transformers`) — no extra key.

In [ ]:
# Dependencies:
# !pip install dspy sentence-transformers orjson python-dotenv
# On Colab: google.colab (preinstalled) supplies the ANTHROPIC_API_KEY from Colab Secrets.

## Part 1 — Configure DSPy with Claude

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()
# On Colab, pull the key from Colab Secrets (sidebar key icon) if it isn't already in the env.
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — add it to your .env file"

# Claude Haiku — fast and cheap, good for experimentation
lm = dspy.LM("anthropic/claude-haiku-4-5-20251001")
dspy.configure(lm=lm)

## Part 2 — Baseline: answer without retrieval

`ChainOfThought` builds the prompt from a typed signature and reasons step by step before
answering — but still only from the model's training memory. This is the baseline retrieval must beat.

In [ ]:
cot = dspy.ChainOfThought("question -> response")
result = cot(question="what are high memory and low memory on linux?")
print(result.response)

In [ ]:
# Inspect the exact prompt and completion DSPy sent to Claude
dspy.inspect_history(n=1)

## Part 3 — Load the dataset

These **question/answer pairs** drive evaluation (Part 4) and prompt optimization (Part 7) — they
are *not* the documents the retriever searches (that corpus is loaded separately in Part 5). Each
example pairs a `question` with a reference `response`; `with_inputs("question")` marks the question
as the input, and the reference answer is a label used only for scoring — never shown to the model
at answer time.

In [ ]:
import orjson
from dspy.utils import download

# The data file already sits next to this notebook. To fetch it from scratch, run this once:
# download("https://huggingface.co/dspy/cache/resolve/main/ragqa_arena_tech_examples.jsonl")

with open("ragqa_arena_tech_examples.jsonl") as f:
    data = [dspy.Example(**orjson.loads(line)).with_inputs("question") for line in f]

print(f"Loaded {len(data)} examples")
data[0]

The dataset has **2,064 examples**; we use the first 1,000, split three ways (the rest go unused — the cap keeps optimization cheap). Each split has a distinct job:

- **`devset` (300)** — the held-out set is used to evaluate and to compare the baseline against the RAG.
- **`trainset` (200)** — used to train a prompt optimizer later on.
- **`testset` (500)** — reserved for a final, unbiased score, in a later notebook.

Keeping them separate stops the optimizer from tuning on the same examples you measure on — which would inflate the numbers.

In [ ]:
import random

random.Random(0).shuffle(data)
trainset, devset, testset = data[:200], data[200:500], data[500:1000]
print(f"Train: {len(trainset)} | Dev: {len(devset)} | Test: {len(testset)}")

## Part 4 — A metric for answer quality

`SemanticF1` scores *meaning* overlap between the predicted and reference answers (not exact
string match), so a correct paraphrase still scores well. It uses the configured LM as a judge.

In [ ]:
from dspy.evaluate import SemanticF1

metric = SemanticF1(decompositional=True)

example = devset[0]
pred = cot(**example.inputs())
score = float(metric(example, pred))

print(f"Question:  {example.question}\n")
print(f"Gold:      {example.response}\n")
print(f"Predicted: {pred.response}\n")
print(f"Semantic F1: {score:.2f}")

In [ ]:
# Baseline CoT across a slice of the dev set.
# num_threads=1 avoids rate-limit errors on starter API tiers; devset[:10] keeps it short.
evaluate = dspy.Evaluate(
    devset=devset[:10],
    metric=metric,
    num_threads=1,
    display_progress=True,
    display_table=2,
)

baseline_score = float(evaluate(cot))
print(f"\nBaseline (CoT, no retrieval): {baseline_score:.1f}%")

## Part 5 — Set up the retriever

Download the corpus of developer-Q&A answers and build a local vector index with
`sentence-transformers` (CPU, no extra key). `max_corpus_size` caps it for a fast demo — the
full corpus is ~28k documents.

**No chunking in this example.** Each corpus document is used **whole** — one document is one
retrievable passage, embedded as-is. These developer-Q&A answers are already short (median ~570
characters / ~100 words; 90% under ~1,900 characters), so they need no splitting; `max_characters =
6000` only trims the rare long outlier (~0.6% of docs). Long-form sources — PDFs, wiki articles —
*do* need chunking; that is §6.

**Loaded into memory.** This cell reads the entire corpus into the `corpus` list in RAM, and the
retriever then holds its embeddings in memory as well — a simple in-process index, not an external
vector database.

In [ ]:
# The data file already sits next to this notebook. To fetch it from scratch, run this once:
# download("https://huggingface.co/dspy/cache/resolve/main/ragqa_arena_tech_corpus.jsonl")

max_characters = 6000     # truncate very long docs
max_corpus_size = 1000    # cap for a fast demo; the full corpus is ~28k docs

with open("ragqa_arena_tech_corpus.jsonl") as f:
    corpus = [orjson.loads(line)["text"][:max_characters]
              for i, line in enumerate(f) if i < max_corpus_size]

print(f"Corpus: {len(corpus)} documents")

**Set up the embedder** — the model that turns text into vectors. We wrap a local
`sentence-transformers` model (`all-MiniLM-L6-v2`, 384-dim, runs on CPU) as a `dspy.Embedder`
callable. The retriever in the next cell uses it to embed both the corpus and each incoming query.

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")

# dspy.Embedder accepts a callable — wrap the model's encode method
def embed(texts):
    return st_model.encode(texts).tolist()

embedder = dspy.Embedder(embed)

**Index the corpus and set up search.** Constructing `dspy.retrievers.Embeddings` **embeds the
entire corpus in one shot** — every document becomes a 384-dim vector, kept in memory. That set of
vectors *is* the index. `k=5` fixes how many passages a query returns. From here, calling
`search(question)` embeds the query and returns its 5 nearest documents by cosine similarity (exact
brute-force at this corpus size — no FAISS).

In [ ]:
search = dspy.retrievers.Embeddings(embedder=embedder, corpus=corpus, k=5)

**Smoke-test the retriever.** Run one query to confirm the index works before wiring it into RAG.
`search(question)` returns a `dspy.Prediction` whose `.passages` are the top-`k` documents; we print
the count and a snippet of the best match to eyeball that retrieval is sane.

In [ ]:
# Smoke test
passages = search("what are high memory and low memory on linux?").passages
print(f"Retrieved {len(passages)} passages. First:\n{passages[0][:300]}")

## Part 5 Full — Set up the retriever, index the full corpus (persisted)

The retriever above indexes only `max_corpus_size = 1000` documents for speed. At ~3.5% of the
corpus, many questions have **no relevant document in the index at all**, so retrieval — and the RAG
answer — suffers. Indexing costs only **CPU, not API tokens** (MiniLM runs locally), so the whole
~28k-doc corpus costs just a one-time ~1–3 min embed. We persist it with DSPy's
`save()` / `from_saved()` so it is not recomputed from run to run.

We keep search **exact** by setting `brute_force_threshold` above the corpus size, rather than
letting DSPy build a lossy FAISS IVFPQ index — brute-force over 28k vectors is instant, needs no
`faiss-cpu`, and retrieves better. (The FAISS/ANN speed–recall tradeoff is its own topic — §5.3, §7.)

To use this index downstream, pass `search_full` instead of `search` when building `RAG` below.

In [ ]:
# Full-corpus index, persisted so it is not recomputed run to run.
INDEX_DIR = "ragqa_index"

if os.path.exists(INDEX_DIR):
    search_full = dspy.retrievers.Embeddings.from_saved(INDEX_DIR, embedder)
    print(f"Loaded prebuilt index from {INDEX_DIR}/ ({len(search_full.corpus)} docs)")
else:
    with open("ragqa_arena_tech_corpus.jsonl") as f:
        full_corpus = [orjson.loads(line)["text"][:max_characters] for line in f]
    # brute_force_threshold above the corpus size keeps search exact (no FAISS dependency)
    search_full = dspy.retrievers.Embeddings(
        embedder=embedder, corpus=full_corpus, k=5, brute_force_threshold=30_000,
    )
    search_full.save(INDEX_DIR)
    print(f"Built and saved index to {INDEX_DIR}/ ({len(search_full.corpus)} docs)")

**Smoke-test the full retriever.** The same query as before, now searching all ~28k documents
(`search_full` reuses the `embedder` from Part 5). Because retrieval draws from the whole corpus,
questions whose relevant document sat outside the first 1,000 can now be found.

In [ ]:
passages = search_full("what are high memory and low memory on linux?").passages
print(f"Retrieved {len(passages)} passages from {len(search_full.corpus):,} docs. First:\n{passages[0][:300]}")

## Part 6 — Build the RAG module

A DSPy module is a Python class. `forward` defines the pipeline: retrieve context, then generate
the answer from context + question.

In [ ]:
class RAG(dspy.Module):
    def __init__(self, retriever):
        self.retriever = retriever
        self.respond = dspy.ChainOfThought("context, question -> response")

    def forward(self, question):
        context = self.retriever(question).passages
        return self.respond(context=context, question=question)


rag = RAG(retriever=search)
print(rag(question="what are high memory and low memory on linux?").response)

**Did retrieval help?** (§4.6) — re-run the same evaluation on the RAG module.

In [ ]:
rag_score = float(evaluate(rag))
print(f"Baseline (no retrieval): {baseline_score:.1f}%")
print(f"RAG:                     {rag_score:.1f}%")

## Part 7 — Optimize the prompt with MIPROv2

MIPROv2 treats the prompt as a variable and searches for better instructions + few-shot demos
against the metric. This backs §8.7. It is **long-running (~15 min)**, so the cell loads a saved
result if one exists and otherwise runs the optimization and saves it.

In [ ]:
if os.path.exists("optimized_rag.json"):
    optimized_rag = RAG(retriever=search)
    optimized_rag.load("optimized_rag.json")
    print("Loaded pre-optimized RAG from optimized_rag.json (skipped optimization)")
else:
    print("No saved result found — running MIPROv2 (~15 min)...")
    optimizer = dspy.MIPROv2(metric=metric, auto="light", num_threads=1)
    optimized_rag = optimizer.compile(
        rag, trainset=trainset,
        max_bootstrapped_demos=2, max_labeled_demos=2,
    )
    optimized_rag.save("optimized_rag.json")

In [ ]:
question = "cmd+tab does not work on hidden or minimized windows"
print("--- Baseline RAG ---")
print(rag(question=question).response)
print("\n--- Optimized RAG ---")
print(optimized_rag(question=question).response)

In [ ]:
optimized_score = float(evaluate(optimized_rag))
print(f"Baseline (no retrieval): {baseline_score:.1f}%")
print(f"RAG:                     {rag_score:.1f}%")
print(f"Optimized RAG:           {optimized_score:.1f}%")

## Part 8 — Cost and persistence

In [ ]:
cost = sum(x["cost"] for x in lm.history if x["cost"] is not None)
print(f"Total LLM cost this session: ${cost:.4f}")

In [ ]:
# Load the saved program back — deploy without re-running optimization.
# (Part 7 already saved it to optimized_rag.json.)
loaded_rag = RAG(retriever=search)
loaded_rag.load("optimized_rag.json")
print(loaded_rag(question="cmd+tab does not work on hidden or minimized windows").response)

## Further reading

- Chapter 04 §3 (build), §4.6 (did retrieval help?), §8.7 (GEPA & MIPROv2).
- `demo04_dspy_gepa.ipynb` — GEPA, the reflective optimizer contrasted with MIPROv2 in §8.7.
- [DSPy RAG tutorial](https://dspy.ai/tutorials/rag/) · [DSPy docs](https://dspy.ai/)